# 🐴 Horsera Perception Engine POC

**Equestrian Biomechanics Analysis using YOLOv8-Pose**

This notebook implements Stages 1-4 of the Horsera perception pipeline:
1. **Video Ingestion & Smart Crop** - Detect horse, crop to preserve detail
2. **Horse Detection** - YOLOv8 to locate horse bounding box
3. **Rider Pose Estimation** - YOLOv8-pose (17 COCO keypoints)
4. **Biomechanic Metric Computation** - 17 metrics from keypoint trajectories

**Outputs:**
- Annotated video with skeleton overlay
- Per-stride metric time-series plots

## Setup & Installation

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python-headless numpy scipy matplotlib pandas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d
from ultralytics import YOLO
from google.colab import files
from IPython.display import HTML, display
import base64
import pandas as pd
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict
import warnings
warnings.filterwarnings('ignore')

print("Dependencies loaded successfully!")

## Upload Video

Upload an equestrian training video (MP4/MOV, ideally 1080p or 4K)

In [ ]:
# Upload video file
print("Please upload your equestrian video file:")
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f"\nUploaded: {VIDEO_PATH}")

## COCO Keypoint Schema (17 points)

```
0: nose          5: left_shoulder   10: right_wrist    15: left_ankle
1: left_eye      6: right_shoulder  11: left_hip       16: right_ankle
2: right_eye     7: left_elbow      12: right_hip
3: left_ear      8: right_elbow     13: left_knee
4: right_ear     9: left_wrist      14: right_knee
```

In [ ]:
# COCO keypoint indices
@dataclass
class COCOKeypoints:
    NOSE = 0
    LEFT_EYE = 1
    RIGHT_EYE = 2
    LEFT_EAR = 3
    RIGHT_EAR = 4
    LEFT_SHOULDER = 5
    RIGHT_SHOULDER = 6
    LEFT_ELBOW = 7
    RIGHT_ELBOW = 8
    LEFT_WRIST = 9
    RIGHT_WRIST = 10
    LEFT_HIP = 11
    RIGHT_HIP = 12
    LEFT_KNEE = 13
    RIGHT_KNEE = 14
    LEFT_ANKLE = 15
    RIGHT_ANKLE = 16

KP = COCOKeypoints()

# Skeleton connections for visualization
SKELETON_CONNECTIONS = [
    (KP.LEFT_ANKLE, KP.LEFT_KNEE),
    (KP.LEFT_KNEE, KP.LEFT_HIP),
    (KP.RIGHT_ANKLE, KP.RIGHT_KNEE),
    (KP.RIGHT_KNEE, KP.RIGHT_HIP),
    (KP.LEFT_HIP, KP.RIGHT_HIP),
    (KP.LEFT_SHOULDER, KP.LEFT_HIP),
    (KP.RIGHT_SHOULDER, KP.RIGHT_HIP),
    (KP.LEFT_SHOULDER, KP.RIGHT_SHOULDER),
    (KP.LEFT_SHOULDER, KP.LEFT_ELBOW),
    (KP.LEFT_ELBOW, KP.LEFT_WRIST),
    (KP.RIGHT_SHOULDER, KP.RIGHT_ELBOW),
    (KP.RIGHT_ELBOW, KP.RIGHT_WRIST),
    (KP.LEFT_EAR, KP.LEFT_SHOULDER),
    (KP.RIGHT_EAR, KP.RIGHT_SHOULDER),
    (KP.NOSE, KP.LEFT_EYE),
    (KP.NOSE, KP.RIGHT_EYE),
    (KP.LEFT_EYE, KP.LEFT_EAR),
    (KP.RIGHT_EYE, KP.RIGHT_EAR),
]

# Color scheme (BGR for OpenCV)
COLORS = {
    'skeleton': (60, 90, 140),      # Cognac brown
    'keypoint': (110, 169, 201),    # Champagne gold
    'occluded': (163, 127, 107),    # Cadence blue (muted)
    'bbox': (118, 155, 125),        # Progress green
}

## Load YOLOv8 Models

In [ ]:
# Load YOLOv8 models
print("Loading YOLOv8 models...")

# YOLOv8m for horse detection (class 17 in COCO = horse)
horse_detector = YOLO('yolov8m.pt')

# YOLOv8m-pose for rider pose estimation
pose_model = YOLO('yolov8m-pose.pt')

print("Models loaded successfully!")

## Stage 1 & 2: Smart Crop Strategy

The smart crop **maximizes resolution on the horse+rider** by:
1. **Pass 1**: Scan video to detect horse bbox across all frames
2. Compute union bounding box (10th-90th percentile for robustness)
3. Expand bbox with 20% padding + extra top padding for rider's head
4. **Pass 2**: Crop frames to the union region and scale up to 640px height
5. Run pose estimation on the cropped region for better keypoint accuracy
6. Apply EMA smoothing (α=0.3) with jump detection for temporal stability

**Result**: Input 640x360 with ~180px rider → Output 640px height with rider filling the frame (~3-4x effective resolution increase)

In [ ]:
class SmartCropper:
    """Implements the Horsera smart crop strategy for 4K video."""
    
    def __init__(self, padding_factor: float = 0.2, ema_alpha: float = 0.3, jump_threshold: float = 0.15):
        self.padding_factor = padding_factor
        self.ema_alpha = ema_alpha
        self.jump_threshold = jump_threshold
        self.prev_bbox = None
        self.smoothed_bbox = None
    
    def detect_horse(self, frame: np.ndarray, detector: YOLO) -> Optional[Tuple[int, int, int, int]]:
        """Detect horse bounding box in frame."""
        # Run detection at reduced resolution for speed
        results = detector(frame, classes=[17], conf=0.3, verbose=False)  # class 17 = horse
        
        if len(results) == 0 or len(results[0].boxes) == 0:
            return None
        
        # Get the largest horse detection (most likely the main subject)
        boxes = results[0].boxes.xyxy.cpu().numpy()
        areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        best_idx = np.argmax(areas)
        
        return tuple(boxes[best_idx].astype(int))
    
    def expand_bbox(self, bbox: Tuple[int, int, int, int], frame_shape: Tuple[int, int]) -> Tuple[int, int, int, int]:
        """Expand bbox by padding factor, ensuring it stays within frame bounds."""
        x1, y1, x2, y2 = bbox
        h, w = frame_shape[:2]
        
        box_w = x2 - x1
        box_h = y2 - y1
        
        # Expand by padding factor
        pad_x = int(box_w * self.padding_factor)
        pad_y = int(box_h * self.padding_factor)
        
        # Extend upward more to capture full rider (1.5x horse back height above)
        pad_y_top = int(box_h * 0.5)  # Extra padding on top for rider
        
        new_x1 = max(0, x1 - pad_x)
        new_y1 = max(0, y1 - pad_y - pad_y_top)
        new_x2 = min(w, x2 + pad_x)
        new_y2 = min(h, y2 + pad_y)
        
        return (new_x1, new_y1, new_x2, new_y2)
    
    def smooth_bbox(self, bbox: Tuple[int, int, int, int], frame_shape: Tuple[int, int]) -> Tuple[int, int, int, int]:
        """Apply EMA smoothing with jump detection."""
        if self.smoothed_bbox is None:
            self.smoothed_bbox = np.array(bbox, dtype=float)
            self.prev_bbox = np.array(bbox, dtype=float)
            return bbox
        
        curr = np.array(bbox, dtype=float)
        h, w = frame_shape[:2]
        
        # Check for abrupt jump (> 15% of frame dimension)
        delta = np.abs(curr - self.prev_bbox)
        frame_dims = np.array([w, h, w, h])
        relative_jump = delta / frame_dims
        
        if np.any(relative_jump > self.jump_threshold):
            # Abrupt jump detected - reset tracking
            self.smoothed_bbox = curr
        else:
            # Apply EMA smoothing
            self.smoothed_bbox = self.ema_alpha * curr + (1 - self.ema_alpha) * self.smoothed_bbox
        
        self.prev_bbox = curr
        return tuple(self.smoothed_bbox.astype(int))
    
    def get_crop_region(self, frame: np.ndarray, detector: YOLO) -> Optional[Tuple[int, int, int, int]]:
        """Get the smart crop region for the frame."""
        horse_bbox = self.detect_horse(frame, detector)
        
        if horse_bbox is None:
            # Fall back to previous bbox if available
            if self.smoothed_bbox is not None:
                return tuple(self.smoothed_bbox.astype(int))
            return None
        
        expanded = self.expand_bbox(horse_bbox, frame.shape)
        smoothed = self.smooth_bbox(expanded, frame.shape)
        
        return smoothed

print("SmartCropper class defined.")

## Stage 3: Rider Pose Estimation

In [ ]:
class RiderPoseEstimator:
    """Estimates rider pose using YOLOv8-pose with mounted rider priority logic."""
    
    def __init__(self, pose_model: YOLO, conf_threshold: float = 0.5):
        self.model = pose_model
        self.conf_threshold = conf_threshold
    
    def estimate_pose(self, frame: np.ndarray, crop_region: Optional[Tuple[int, int, int, int]] = None) -> Optional[np.ndarray]:
        """
        Estimate rider pose in the frame.
        
        Returns:
            keypoints: (17, 3) array of [x, y, confidence] for each keypoint
        """
        # Use crop region if provided
        if crop_region is not None:
            x1, y1, x2, y2 = crop_region
            cropped = frame[y1:y2, x1:x2]
            offset = np.array([x1, y1])
        else:
            cropped = frame
            offset = np.array([0, 0])
        
        # Run pose estimation
        results = self.model(cropped, conf=self.conf_threshold, verbose=False)
        
        if len(results) == 0 or results[0].keypoints is None:
            return None
        
        keypoints_data = results[0].keypoints
        
        if keypoints_data.xy is None or len(keypoints_data.xy) == 0:
            return None
        
        # Get keypoints and confidences
        kps_xy = keypoints_data.xy.cpu().numpy()  # (N, 17, 2)
        kps_conf = keypoints_data.conf.cpu().numpy() if keypoints_data.conf is not None else np.ones((len(kps_xy), 17))
        
        # Apply mounted rider priority logic:
        # Select person whose hip centroid is in the upper portion of the crop
        if len(kps_xy) > 1:
            crop_h = y2 - y1 if crop_region else frame.shape[0]
            best_idx = 0
            best_score = float('inf')
            
            for i, kps in enumerate(kps_xy):
                # Get hip centroid
                left_hip = kps[KP.LEFT_HIP]
                right_hip = kps[KP.RIGHT_HIP]
                hip_y = (left_hip[1] + right_hip[1]) / 2
                
                # Prefer person in upper 40% of crop (mounted rider)
                if hip_y < crop_h * 0.6:  # Upper 60% to be safe
                    # Score by how high up they are
                    if hip_y < best_score:
                        best_score = hip_y
                        best_idx = i
            
            kps_xy = kps_xy[best_idx:best_idx+1]
            kps_conf = kps_conf[best_idx:best_idx+1]
        
        # Combine xy and confidence into (17, 3) array
        keypoints = np.zeros((17, 3))
        keypoints[:, :2] = kps_xy[0] + offset  # Add offset to get full-frame coordinates
        keypoints[:, 2] = kps_conf[0]
        
        return keypoints

print("RiderPoseEstimator class defined.")

## Stage 4: Biomechanic Metrics

Computing 17 metrics available with COCO 17-keypoint schema:

| Group | Metrics |
|-------|--------|
| Lower Leg Stability | Lower Leg Stability, Knee Angle Stability, Lower Leg Drift |
| Hand/Rein Control | Rein Symmetry, Rein Steadiness, Hand Spacing Stability, Elbow Symmetry, Elbow Elasticity |
| Upper Body Alignment | Upper Body Vertical Alignment, Shoulder Levelness, Head Stability |
| Core & Seat | Core Stability, Trunk Angle Stability, Pelvis Vertical Stability, Pelvis Levelness |
| Balance/Symmetry | Rider Centerline Alignment, Left-Right Symmetry Index |

In [ ]:
class BiomechanicsCalculator:
    """Computes equestrian biomechanic metrics from keypoint trajectories."""
    
    def __init__(self, fps: float = 30.0):
        self.fps = fps
        self.keypoint_history: List[np.ndarray] = []
        self.metrics_history: Dict[str, List[float]] = {}
    
    def add_frame(self, keypoints: np.ndarray):
        """Add a frame's keypoints to the history."""
        self.keypoint_history.append(keypoints.copy())
    
    def _get_point(self, keypoints: np.ndarray, idx: int) -> np.ndarray:
        """Get (x, y) position for a keypoint."""
        return keypoints[idx, :2]
    
    def _get_conf(self, keypoints: np.ndarray, idx: int) -> float:
        """Get confidence for a keypoint."""
        return keypoints[idx, 2]
    
    def _angle(self, p1: np.ndarray, p2: np.ndarray, p3: np.ndarray) -> float:
        """Calculate angle at p2 formed by p1-p2-p3 in degrees."""
        v1 = p1 - p2
        v2 = p3 - p2
        
        cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
        cos_angle = np.clip(cos_angle, -1, 1)
        
        return np.degrees(np.arccos(cos_angle))
    
    def _angle_to_vertical(self, p1: np.ndarray, p2: np.ndarray) -> float:
        """Calculate angle of vector p1->p2 relative to vertical (positive = leaning forward)."""
        vec = p2 - p1  # From bottom to top (hip to shoulder)
        # Vertical is (0, -1) in image coordinates (y increases downward)
        vertical = np.array([0, -1])
        
        cos_angle = np.dot(vec, vertical) / (np.linalg.norm(vec) + 1e-8)
        cos_angle = np.clip(cos_angle, -1, 1)
        angle = np.degrees(np.arccos(cos_angle))
        
        # Sign: positive if leaning forward (shoulder ahead of hip in x)
        if vec[0] > 0:
            angle = -angle
        
        return angle
    
    def _torso_length(self, keypoints: np.ndarray) -> float:
        """Calculate torso length (mid-shoulder to mid-hip)."""
        mid_shoulder = (self._get_point(keypoints, KP.LEFT_SHOULDER) + 
                       self._get_point(keypoints, KP.RIGHT_SHOULDER)) / 2
        mid_hip = (self._get_point(keypoints, KP.LEFT_HIP) + 
                  self._get_point(keypoints, KP.RIGHT_HIP)) / 2
        return np.linalg.norm(mid_shoulder - mid_hip) + 1e-8
    
    def _shoulder_width(self, keypoints: np.ndarray) -> float:
        """Calculate shoulder width."""
        return np.linalg.norm(
            self._get_point(keypoints, KP.LEFT_SHOULDER) - 
            self._get_point(keypoints, KP.RIGHT_SHOULDER)
        ) + 1e-8
    
    def _hip_width(self, keypoints: np.ndarray) -> float:
        """Calculate hip width."""
        return np.linalg.norm(
            self._get_point(keypoints, KP.LEFT_HIP) - 
            self._get_point(keypoints, KP.RIGHT_HIP)
        ) + 1e-8
    
    def compute_all_metrics(self) -> Dict[str, float]:
        """Compute all biomechanic metrics from the keypoint history."""
        if len(self.keypoint_history) < 10:
            return {}
        
        kps = np.array(self.keypoint_history)  # (T, 17, 3)
        T = len(kps)
        
        metrics = {}
        
        # === LOWER LEG STABILITY GROUP ===
        
        # 1. Lower Leg Stability: RMS of ankle-hip vector deviation / torso_length
        ankle_l = kps[:, KP.LEFT_ANKLE, :2]
        ankle_r = kps[:, KP.RIGHT_ANKLE, :2]
        hip_l = kps[:, KP.LEFT_HIP, :2]
        hip_r = kps[:, KP.RIGHT_HIP, :2]
        
        v_leg_l = ankle_l - hip_l
        v_leg_r = ankle_r - hip_r
        v_leg = (v_leg_l + v_leg_r) / 2
        v_leg_mean = np.mean(v_leg, axis=0)
        v_leg_dev = v_leg - v_leg_mean
        
        torso_lengths = np.array([self._torso_length(k) for k in kps])
        mean_torso = np.mean(torso_lengths)
        
        lower_leg_stability = np.sqrt(np.mean(np.sum(v_leg_dev**2, axis=1))) / mean_torso
        metrics['lower_leg_stability'] = lower_leg_stability
        
        # 2. Knee Angle Stability: StdDev of knee angle over time
        knee_angles_l = np.array([self._angle(
            self._get_point(k, KP.LEFT_HIP),
            self._get_point(k, KP.LEFT_KNEE),
            self._get_point(k, KP.LEFT_ANKLE)
        ) for k in kps])
        
        knee_angles_r = np.array([self._angle(
            self._get_point(k, KP.RIGHT_HIP),
            self._get_point(k, KP.RIGHT_KNEE),
            self._get_point(k, KP.RIGHT_ANKLE)
        ) for k in kps])
        
        knee_angle_stability = (np.std(knee_angles_l) + np.std(knee_angles_r)) / 2
        metrics['knee_angle_stability'] = knee_angle_stability
        
        # 3. Lower Leg Drift: Mean deviation from baseline position
        baseline_leg = v_leg[0]  # First frame as baseline
        leg_drift = np.mean(np.linalg.norm(np.mean(v_leg, axis=0) - baseline_leg)) / mean_torso
        metrics['lower_leg_drift'] = leg_drift
        
        # === HAND / REIN CONTROL GROUP ===
        
        # 4. Rein Symmetry: |y(LW) - y(RW)| / shoulder_width
        wrist_l = kps[:, KP.LEFT_WRIST, :2]
        wrist_r = kps[:, KP.RIGHT_WRIST, :2]
        
        shoulder_widths = np.array([self._shoulder_width(k) for k in kps])
        mean_shoulder_width = np.mean(shoulder_widths)
        
        wrist_y_diff = np.abs(np.mean(wrist_l[:, 1]) - np.mean(wrist_r[:, 1]))
        rein_symmetry = wrist_y_diff / mean_shoulder_width
        metrics['rein_symmetry'] = rein_symmetry
        
        # 5. Rein Steadiness: RMS of hand movement relative to shoulder / torso_length
        shoulder_l = kps[:, KP.LEFT_SHOULDER, :2]
        shoulder_r = kps[:, KP.RIGHT_SHOULDER, :2]
        
        v_hand_l = wrist_l - shoulder_l
        v_hand_r = wrist_r - shoulder_r
        v_hand = (v_hand_l + v_hand_r) / 2
        v_hand_mean = np.mean(v_hand, axis=0)
        v_hand_dev = v_hand - v_hand_mean
        
        rein_steadiness = np.sqrt(np.mean(np.sum(v_hand_dev**2, axis=1))) / mean_torso
        metrics['rein_steadiness'] = rein_steadiness
        
        # 6. Hand Spacing Stability: StdDev of hand distance / shoulder_width
        hand_distances = np.linalg.norm(wrist_l - wrist_r, axis=1)
        hand_spacing_stability = np.std(hand_distances / mean_shoulder_width)
        metrics['hand_spacing_stability'] = hand_spacing_stability
        
        # 7. Elbow Symmetry: |theta_elbow_left - theta_elbow_right|
        elbow_angles_l = np.array([self._angle(
            self._get_point(k, KP.LEFT_SHOULDER),
            self._get_point(k, KP.LEFT_ELBOW),
            self._get_point(k, KP.LEFT_WRIST)
        ) for k in kps])
        
        elbow_angles_r = np.array([self._angle(
            self._get_point(k, KP.RIGHT_SHOULDER),
            self._get_point(k, KP.RIGHT_ELBOW),
            self._get_point(k, KP.RIGHT_WRIST)
        ) for k in kps])
        
        elbow_symmetry = np.abs(np.mean(elbow_angles_l) - np.mean(elbow_angles_r))
        metrics['elbow_symmetry'] = elbow_symmetry
        
        # 8. Elbow Elasticity: Mean and StdDev of elbow angle
        mean_elbow_angle = (np.mean(elbow_angles_l) + np.mean(elbow_angles_r)) / 2
        elbow_variability = (np.std(elbow_angles_l) + np.std(elbow_angles_r)) / 2
        metrics['elbow_elasticity_mean'] = mean_elbow_angle
        metrics['elbow_elasticity_var'] = elbow_variability
        
        # === UPPER BODY ALIGNMENT GROUP ===
        
        # 9. Upper Body Vertical Alignment: Mean trunk angle relative to vertical
        trunk_angles = np.array([self._angle_to_vertical(
            (self._get_point(k, KP.LEFT_HIP) + self._get_point(k, KP.RIGHT_HIP)) / 2,
            (self._get_point(k, KP.LEFT_SHOULDER) + self._get_point(k, KP.RIGHT_SHOULDER)) / 2
        ) for k in kps])
        
        upper_body_alignment = np.mean(np.abs(trunk_angles))
        metrics['upper_body_vertical_alignment'] = upper_body_alignment
        
        # 10. Shoulder Levelness: |y(LS) - y(RS)| / shoulder_width
        shoulder_y_diff = np.abs(np.mean(shoulder_l[:, 1]) - np.mean(shoulder_r[:, 1]))
        shoulder_levelness = shoulder_y_diff / mean_shoulder_width
        metrics['shoulder_levelness'] = shoulder_levelness
        
        # 11. Head Stability: RMS of head movement / torso_length
        # Using nose as head proxy
        head = kps[:, KP.NOSE, :2]
        head_mean = np.mean(head, axis=0)
        head_dev = head - head_mean
        head_stability = np.sqrt(np.mean(np.sum(head_dev**2, axis=1))) / mean_torso
        metrics['head_stability'] = head_stability
        
        # === CORE & SEAT STABILITY GROUP ===
        
        # 12. Core Stability: RMS of trunk vector deviation / torso_length
        mid_shoulder = (shoulder_l + shoulder_r) / 2
        mid_hip = (hip_l + hip_r) / 2
        trunk_vector = mid_shoulder - mid_hip
        trunk_mean = np.mean(trunk_vector, axis=0)
        trunk_dev = trunk_vector - trunk_mean
        
        core_stability = np.sqrt(np.mean(np.sum(trunk_dev**2, axis=1))) / mean_torso
        metrics['core_stability'] = core_stability
        
        # 13. Trunk Angle Stability: StdDev of trunk angle
        trunk_angle_stability = np.std(trunk_angles)
        metrics['trunk_angle_stability'] = trunk_angle_stability
        
        # 14. Pelvis Vertical Stability: StdDev of mid-hip y / torso_length
        pelvis_vertical_stability = np.std(mid_hip[:, 1]) / mean_torso
        metrics['pelvis_vertical_stability'] = pelvis_vertical_stability
        
        # 15. Pelvis Levelness: |y(LH) - y(RH)| / hip_width
        hip_widths = np.array([self._hip_width(k) for k in kps])
        mean_hip_width = np.mean(hip_widths)
        
        hip_y_diff = np.abs(np.mean(hip_l[:, 1]) - np.mean(hip_r[:, 1]))
        pelvis_levelness = hip_y_diff / mean_hip_width
        metrics['pelvis_levelness'] = pelvis_levelness
        
        # === BALANCE / SYMMETRY GROUP ===
        
        # 16. Rider Centerline Alignment: |x(MS) - x(MH)| / torso_length
        centerline_x_diff = np.abs(np.mean(mid_shoulder[:, 0]) - np.mean(mid_hip[:, 0]))
        rider_centerline = centerline_x_diff / mean_torso
        metrics['rider_centerline_alignment'] = rider_centerline
        
        # 17. Left-Right Symmetry Index: Mean of shoulder_levelness, pelvis_levelness, knee_asymmetry
        knee_asymmetry = np.abs(np.mean(knee_angles_l) - np.mean(knee_angles_r)) / 180.0  # Normalize
        symmetry_index = np.mean([shoulder_levelness, pelvis_levelness, knee_asymmetry])
        metrics['left_right_symmetry_index'] = symmetry_index
        
        return metrics
    
    def compute_per_frame_metrics(self) -> pd.DataFrame:
        """Compute per-frame metrics for time-series plotting."""
        if len(self.keypoint_history) < 2:
            return pd.DataFrame()
        
        kps = np.array(self.keypoint_history)
        T = len(kps)
        
        frames = []
        
        for t, k in enumerate(kps):
            frame_metrics = {'frame': t, 'time': t / self.fps}
            
            # Trunk angle
            mid_hip = (self._get_point(k, KP.LEFT_HIP) + self._get_point(k, KP.RIGHT_HIP)) / 2
            mid_shoulder = (self._get_point(k, KP.LEFT_SHOULDER) + self._get_point(k, KP.RIGHT_SHOULDER)) / 2
            frame_metrics['trunk_angle'] = self._angle_to_vertical(mid_hip, mid_shoulder)
            
            # Knee angles
            frame_metrics['knee_angle_left'] = self._angle(
                self._get_point(k, KP.LEFT_HIP),
                self._get_point(k, KP.LEFT_KNEE),
                self._get_point(k, KP.LEFT_ANKLE)
            )
            frame_metrics['knee_angle_right'] = self._angle(
                self._get_point(k, KP.RIGHT_HIP),
                self._get_point(k, KP.RIGHT_KNEE),
                self._get_point(k, KP.RIGHT_ANKLE)
            )
            
            # Elbow angles
            frame_metrics['elbow_angle_left'] = self._angle(
                self._get_point(k, KP.LEFT_SHOULDER),
                self._get_point(k, KP.LEFT_ELBOW),
                self._get_point(k, KP.LEFT_WRIST)
            )
            frame_metrics['elbow_angle_right'] = self._angle(
                self._get_point(k, KP.RIGHT_SHOULDER),
                self._get_point(k, KP.RIGHT_ELBOW),
                self._get_point(k, KP.RIGHT_WRIST)
            )
            
            # Pelvis y-position (for stride detection)
            frame_metrics['pelvis_y'] = mid_hip[1]
            
            # Shoulder levelness
            shoulder_width = self._shoulder_width(k)
            shoulder_y_diff = np.abs(
                self._get_point(k, KP.LEFT_SHOULDER)[1] - 
                self._get_point(k, KP.RIGHT_SHOULDER)[1]
            )
            frame_metrics['shoulder_levelness'] = shoulder_y_diff / shoulder_width
            
            frames.append(frame_metrics)
        
        return pd.DataFrame(frames)

print("BiomechanicsCalculator class defined.")

## Video Processing Pipeline

In [ ]:
def draw_skeleton(frame: np.ndarray, keypoints: np.ndarray,
                  conf_threshold: float = 0.3,
                  offset: Tuple[int, int] = (0, 0),
                  scale: float = 1.0) -> np.ndarray:
    """Draw skeleton overlay on frame.

    Args:
        frame: Image to draw on
        keypoints: (17, 3) array of [x, y, confidence] in original frame coordinates
        conf_threshold: Minimum confidence to draw keypoint
        offset: (x, y) offset to subtract from keypoints (for cropped output)
        scale: Scale factor to apply after offset (for resized output)
    """
    frame_out = frame.copy()

    # Adjust keypoints for crop offset and scale
    adjusted_kps = keypoints.copy()
    adjusted_kps[:, 0] = (keypoints[:, 0] - offset[0]) * scale
    adjusted_kps[:, 1] = (keypoints[:, 1] - offset[1]) * scale

    # Fixed sizes - don't scale with zoom to keep annotations clean
    thickness = 2
    radius = 4

    # Draw skeleton connections
    for (i, j) in SKELETON_CONNECTIONS:
        if keypoints[i, 2] > conf_threshold and keypoints[j, 2] > conf_threshold:
            pt1 = tuple(adjusted_kps[i, :2].astype(int))
            pt2 = tuple(adjusted_kps[j, :2].astype(int))
            cv2.line(frame_out, pt1, pt2, COLORS['skeleton'], thickness, cv2.LINE_AA)

    # Draw keypoints
    for i in range(17):
        conf = keypoints[i, 2]
        if conf > conf_threshold:
            pt = tuple(adjusted_kps[i, :2].astype(int))
            color = COLORS['keypoint'] if conf > 0.5 else COLORS['occluded']
            cv2.circle(frame_out, pt, radius, color, -1, cv2.LINE_AA)
            cv2.circle(frame_out, pt, radius + 1, (255, 255, 255), 1, cv2.LINE_AA)

    return frame_out


# Output resolution for cropped video (maintains aspect ratio)
OUTPUT_HEIGHT = 640  # Target height in pixels


def process_video(video_path: str, max_frames: int = None) -> Tuple[str, Dict, pd.DataFrame]:
    """
    Process video through the full perception pipeline.

    The output video is CROPPED to the horse+rider region and scaled up
    to maximize resolution on the subject (per PRD Smart Crop Strategy).

    Returns:
        output_path: Path to annotated video
        metrics: Aggregate biomechanic metrics
        per_frame_df: Per-frame metrics dataframe
    """
    # Open video
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if max_frames:
        total_frames = min(total_frames, max_frames)

    print(f"Input video: {width}x{height} @ {fps:.1f} fps, {total_frames} frames")

    # Initialize components
    cropper = SmartCropper()
    pose_estimator = RiderPoseEstimator(pose_model)
    biomech = BiomechanicsCalculator(fps=fps)

    # First pass: detect crop regions to determine output size
    print("Pass 1: Detecting horse+rider regions...")
    crop_regions = []
    cap_temp = cv2.VideoCapture(video_path)
    temp_count = 0
    while cap_temp.isOpened() and temp_count < total_frames:
        ret, frame = cap_temp.read()
        if not ret:
            break
        crop_region = cropper.get_crop_region(frame, horse_detector)
        if crop_region is not None:
            crop_regions.append(crop_region)
        temp_count += 1
        if temp_count % 60 == 0:
            print(f"  Scanned {temp_count}/{total_frames} frames")
    cap_temp.release()

    if len(crop_regions) == 0:
        print("ERROR: No horse detected in video!")
        return None, {}, pd.DataFrame()

    # Calculate the union bounding box of all crop regions (for consistent output size)
    crop_regions = np.array(crop_regions)
    union_x1 = int(np.percentile(crop_regions[:, 0], 10))  # Use 10th percentile to be robust
    union_y1 = int(np.percentile(crop_regions[:, 1], 10))
    union_x2 = int(np.percentile(crop_regions[:, 2], 90))
    union_y2 = int(np.percentile(crop_regions[:, 3], 90))

    # Ensure bounds are valid
    union_x1 = max(0, union_x1)
    union_y1 = max(0, union_y1)
    union_x2 = min(width, union_x2)
    union_y2 = min(height, union_y2)

    crop_w = union_x2 - union_x1
    crop_h = union_y2 - union_y1

    # Calculate output dimensions (scale to OUTPUT_HEIGHT, maintain aspect ratio)
    scale = OUTPUT_HEIGHT / crop_h
    output_w = int(crop_w * scale)
    output_h = OUTPUT_HEIGHT

    # Ensure even dimensions for video codec
    output_w = output_w + (output_w % 2)
    output_h = output_h + (output_h % 2)

    print(f"Crop region: ({union_x1}, {union_y1}) to ({union_x2}, {union_y2}) = {crop_w}x{crop_h}")
    print(f"Output video: {output_w}x{output_h} (scale: {scale:.2f}x)")

    # Setup output video
    output_path = 'annotated_output.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (output_w, output_h))

    # Reset cropper for second pass
    cropper = SmartCropper()

    frame_count = 0
    successful_detections = 0

    print("\nPass 2: Processing frames with pose estimation...")

    while cap.isOpened() and frame_count < total_frames:
        ret, frame = cap.read()
        if not ret:
            break

        # Stage 1-2: Smart crop (for pose estimation)
        crop_region = cropper.get_crop_region(frame, horse_detector)

        # Stage 3: Pose estimation on the dynamic crop
        keypoints = pose_estimator.estimate_pose(frame, crop_region)

        # Extract the fixed output crop region from the frame
        crop_frame = frame[union_y1:union_y2, union_x1:union_x2]

        # Resize to output dimensions
        output_frame = cv2.resize(crop_frame, (output_w, output_h), interpolation=cv2.INTER_LINEAR)

        if keypoints is not None:
            # Stage 4: Add to biomechanics calculator (using original coordinates)
            biomech.add_frame(keypoints)
            successful_detections += 1

            # Draw skeleton overlay (adjusted for crop offset and scale)
            output_frame = draw_skeleton(
                output_frame,
                keypoints,
                offset=(union_x1, union_y1),
                scale=scale
            )

        out.write(output_frame)
        frame_count += 1

        if frame_count % 30 == 0:
            print(f"  Processed {frame_count}/{total_frames} frames ({100*frame_count/total_frames:.1f}%)")

    cap.release()
    out.release()

    print(f"\nProcessing complete!")
    print(f"  Successful detections: {successful_detections}/{frame_count} ({100*successful_detections/frame_count:.1f}%)")
    print(f"  Resolution increase: {crop_h}px → {output_h}px ({scale:.1f}x zoom)")

    # Compute metrics
    metrics = biomech.compute_all_metrics()
    per_frame_df = biomech.compute_per_frame_metrics()

    return output_path, metrics, per_frame_df

print("Video processing functions defined.")

## Run Pipeline

In [ ]:
# Process the video (limit to first 300 frames for quick testing)
output_path, metrics, per_frame_df = process_video(VIDEO_PATH, max_frames=300)

## View Annotated Video

In [ ]:
# Display video in notebook
def show_video(video_path: str):
    """Display video in Colab notebook."""
    from IPython.display import HTML
    import base64
    
    with open(video_path, 'rb') as f:
        video_data = f.read()
    
    video_b64 = base64.b64encode(video_data).decode()
    
    html = f'''
    <video width="640" controls>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
    </video>
    '''
    display(HTML(html))

print("Annotated video with skeleton overlay:")
show_video(output_path)

## Biomechanic Metrics Summary

In [ ]:
# Display metrics with feedback mapping
FEEDBACK_MAPPING = {
    'lower_leg_stability': {
        'name': 'Lower Leg Stability',
        'ideal': 'Close to 0',
        'thresholds': [(0.03, 'Excellent'), (0.06, 'Very stable'), (0.10, 'Slight movement'), (0.15, 'Instability visible')]
    },
    'knee_angle_stability': {
        'name': 'Knee Angle Stability',
        'ideal': '0° variation',
        'thresholds': [(3, 'Very stable'), (6, 'Mostly stable'), (10, 'Mild movement'), (18, 'Noticeable instability')]
    },
    'lower_leg_drift': {
        'name': 'Lower Leg Drift',
        'ideal': '0',
        'thresholds': [(0.02, 'Ideal'), (0.05, 'Slight drift'), (0.09, 'Moderate drift'), (0.12, 'Creeping forward/back')]
    },
    'rein_symmetry': {
        'name': 'Rein Symmetry',
        'ideal': '0',
        'thresholds': [(0.02, 'Perfectly even'), (0.05, 'Mostly level'), (0.08, 'Slight imbalance'), (0.15, 'Noticeable uneven')]
    },
    'rein_steadiness': {
        'name': 'Rein Steadiness',
        'ideal': '0',
        'thresholds': [(0.02, 'Extremely steady'), (0.05, 'Steady'), (0.08, 'Slight movement'), (0.15, 'Moving frequently')]
    },
    'hand_spacing_stability': {
        'name': 'Hand Spacing Stability',
        'ideal': '0',
        'thresholds': [(0.02, 'Very consistent'), (0.04, 'Mostly consistent'), (0.06, 'Mild inconsistency'), (0.10, 'Noticeably variable')]
    },
    'elbow_symmetry': {
        'name': 'Elbow Symmetry',
        'ideal': '0°',
        'thresholds': [(3, 'Very even'), (6, 'Mostly even'), (10, 'Mild asymmetry'), (18, 'Noticeable asymmetry')]
    },
    'upper_body_vertical_alignment': {
        'name': 'Upper Body Alignment',
        'ideal': '0°',
        'thresholds': [(5, 'Excellent posture'), (10, 'Good alignment'), (15, 'Slight lean'), (25, 'Noticeable lean')]
    },
    'shoulder_levelness': {
        'name': 'Shoulder Levelness',
        'ideal': '0',
        'thresholds': [(0.02, 'Level'), (0.05, 'Minor tilt'), (0.08, 'Slight lean'), (0.15, 'Noticeable tilt')]
    },
    'head_stability': {
        'name': 'Head Stability',
        'ideal': '0',
        'thresholds': [(0.02, 'Very stable'), (0.04, 'Mostly steady'), (0.07, 'Mild movement'), (0.12, 'Noticeable movement')]
    },
    'core_stability': {
        'name': 'Core Stability',
        'ideal': '0',
        'thresholds': [(0.015, 'Excellent'), (0.035, 'Good'), (0.07, 'Slight instability'), (0.12, 'Unstable')]
    },
    'trunk_angle_stability': {
        'name': 'Trunk Angle Stability',
        'ideal': '0°',
        'thresholds': [(2, 'Extremely stable'), (4, 'Stable'), (7, 'Mild variation'), (12, 'Noticeable variation')]
    },
    'pelvis_vertical_stability': {
        'name': 'Pelvis Vertical Stability',
        'ideal': '0.01-0.03 (should move with horse)',
        'thresholds': [(0.03, 'Excellent seat'), (0.05, 'Smooth seat'), (0.08, 'Slight bounce'), (0.12, 'Noticeable bounce')]
    },
    'pelvis_levelness': {
        'name': 'Pelvis Levelness',
        'ideal': '0',
        'thresholds': [(0.02, 'Level'), (0.05, 'Slight asymmetry'), (0.08, 'Mild collapse'), (0.15, 'Noticeable unevenness')]
    },
    'rider_centerline_alignment': {
        'name': 'Rider Centerline',
        'ideal': '0',
        'thresholds': [(0.02, 'Centered'), (0.05, 'Slightly off-center'), (0.08, 'Moderate offset'), (0.12, 'Noticeable imbalance')]
    },
    'left_right_symmetry_index': {
        'name': 'L/R Symmetry Index',
        'ideal': '0',
        'thresholds': [(0.03, 'Highly symmetrical'), (0.06, 'Mostly symmetrical'), (0.10, 'Mild asymmetry'), (0.15, 'Noticeable asymmetry')]
    },
}

def get_feedback(metric_key: str, value: float) -> str:
    """Get feedback text for a metric value."""
    if metric_key not in FEEDBACK_MAPPING:
        return "N/A"
    
    thresholds = FEEDBACK_MAPPING[metric_key]['thresholds']
    for threshold, feedback in thresholds:
        if value <= threshold:
            return feedback
    return thresholds[-1][1]  # Return last feedback if above all thresholds

print("\n" + "="*60)
print("🐴 HORSERA BIOMECHANIC METRICS REPORT")
print("="*60 + "\n")

for key, value in metrics.items():
    if key in FEEDBACK_MAPPING:
        info = FEEDBACK_MAPPING[key]
        feedback = get_feedback(key, value)
        print(f"{info['name']:30} {value:8.4f}  →  {feedback}")
    elif key == 'elbow_elasticity_mean':
        print(f"{'Elbow Elasticity (Mean)':30} {value:8.1f}°")
    elif key == 'elbow_elasticity_var':
        print(f"{'Elbow Elasticity (Variation)':30} {value:8.1f}°")

print("\n" + "="*60)

## Per-Stride Metric Time-Series

In [ ]:
# Apply temporal smoothing and plot metrics
if len(per_frame_df) > 0:
    # Smooth the data with Savitzky-Golay filter
    window = min(15, len(per_frame_df) // 2 * 2 - 1)  # Must be odd
    if window >= 5:
        for col in ['trunk_angle', 'knee_angle_left', 'knee_angle_right', 
                    'elbow_angle_left', 'elbow_angle_right', 'pelvis_y', 'shoulder_levelness']:
            if col in per_frame_df.columns:
                per_frame_df[f'{col}_smooth'] = savgol_filter(per_frame_df[col], window, 2)
    
    # Create figure with subplots
    fig, axes = plt.subplots(4, 1, figsize=(14, 12))
    fig.suptitle('🐴 Horsera Biomechanic Time Series', fontsize=14, fontweight='bold')
    
    time = per_frame_df['time']
    
    # Plot 1: Trunk Angle
    ax1 = axes[0]
    ax1.plot(time, per_frame_df['trunk_angle'], alpha=0.3, color='#8C5A3C', label='Raw')
    if 'trunk_angle_smooth' in per_frame_df.columns:
        ax1.plot(time, per_frame_df['trunk_angle_smooth'], color='#8C5A3C', linewidth=2, label='Smoothed')
    ax1.axhline(y=0, color='#7D9B76', linestyle='--', alpha=0.5, label='Ideal (vertical)')
    ax1.axhspan(-5, 5, alpha=0.1, color='#7D9B76', label='Target range')
    ax1.set_ylabel('Trunk Angle (°)')
    ax1.set_title('Upper Body Vertical Alignment')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Knee Angles
    ax2 = axes[1]
    ax2.plot(time, per_frame_df['knee_angle_left'], alpha=0.3, color='#6B7FA3')
    ax2.plot(time, per_frame_df['knee_angle_right'], alpha=0.3, color='#C9A96E')
    if 'knee_angle_left_smooth' in per_frame_df.columns:
        ax2.plot(time, per_frame_df['knee_angle_left_smooth'], color='#6B7FA3', linewidth=2, label='Left Knee')
        ax2.plot(time, per_frame_df['knee_angle_right_smooth'], color='#C9A96E', linewidth=2, label='Right Knee')
    ax2.axhspan(100, 130, alpha=0.1, color='#7D9B76', label='Target range')
    ax2.set_ylabel('Knee Angle (°)')
    ax2.set_title('Knee Angles (L/R)')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Elbow Angles
    ax3 = axes[2]
    ax3.plot(time, per_frame_df['elbow_angle_left'], alpha=0.3, color='#6B7FA3')
    ax3.plot(time, per_frame_df['elbow_angle_right'], alpha=0.3, color='#C9A96E')
    if 'elbow_angle_left_smooth' in per_frame_df.columns:
        ax3.plot(time, per_frame_df['elbow_angle_left_smooth'], color='#6B7FA3', linewidth=2, label='Left Elbow')
        ax3.plot(time, per_frame_df['elbow_angle_right_smooth'], color='#C9A96E', linewidth=2, label='Right Elbow')
    ax3.axhspan(130, 155, alpha=0.1, color='#7D9B76', label='Elastic target')
    ax3.set_ylabel('Elbow Angle (°)')
    ax3.set_title('Elbow Angles (L/R) - Rein Contact')
    ax3.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Pelvis Y (for stride detection)
    ax4 = axes[3]
    pelvis_y = per_frame_df['pelvis_y']
    pelvis_normalized = (pelvis_y - pelvis_y.mean()) / pelvis_y.std()
    ax4.plot(time, pelvis_normalized, alpha=0.3, color='#8C5A3C')
    if 'pelvis_y_smooth' in per_frame_df.columns:
        pelvis_smooth_norm = (per_frame_df['pelvis_y_smooth'] - pelvis_y.mean()) / pelvis_y.std()
        ax4.plot(time, pelvis_smooth_norm, color='#8C5A3C', linewidth=2, label='Pelvis Position')
    ax4.set_ylabel('Pelvis Y (normalized)')
    ax4.set_xlabel('Time (seconds)')
    ax4.set_title('Pelvis Vertical Movement (Stride Pattern)')
    ax4.legend(loc='upper right')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('biomechanic_timeseries.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nTime-series plot saved to 'biomechanic_timeseries.png'")
else:
    print("Not enough frames to generate time-series plots.")

## Download Results

In [ ]:
# Download annotated video and plots
print("Downloading results...")

files.download(output_path)
files.download('biomechanic_timeseries.png')

# Export metrics to CSV
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv('biomechanic_metrics.csv', index=False)
files.download('biomechanic_metrics.csv')

print("\n✅ All results downloaded!")

---

## Notes

### Metrics Not Available with COCO 17-Keypoint Schema

The following metrics require foot landmarks (heel, toe) not present in COCO:
- Heel Position Stability
- Heel Depth
- Ankle Flexion Angle
- Foot Stability

To enable these, upgrade to RTMPose-l with 133-keypoint wholebody schema.

### Next Steps for Production

1. **Fine-tune horse detector** on equestrian footage for better mounted-horse specificity
2. **Upgrade to RTMPose wholebody** for heel/foot metrics
3. **Add stride cycle detection** for per-stride metric aggregation
4. **CoreML export** for on-device inference on iOS